In [1]:
import pandas as pd
from src.pipeline.rul_pipeline import RULPipeline

df = pd.read_csv('data/clean/data_motor_1.csv')
df.insert(0, 'unit_number', 1)

pipeline = RULPipeline(window_size=30, clipping_threshold=125, verbose=True)
motor_windows = pipeline.transform(df)

print(f"\nMotor 1:")
print(f"  X_windows shape: {motor_windows[1]['X_windows'].shape}")
print(f"  Eventos: {motor_windows[1]['evento'].sum()}")

[Nodo 2] 1 motors, 163 windows — 0.01s
[Nodo 3] 192 features per window — 2.88s
[Total]  2.89s

Motor 1:
  X_windows shape: (163, 192)
  Eventos: 1


In [2]:
import pandas as pd
import time
from src.dataset_manager import DatasetManager
from src.pipeline.rul_pipeline import RULPipeline

m_train, _ = DatasetManager.split_dataset()

# Cargar todos los motores de training
dfs = []
for idx in m_train:
    df_motor = pd.read_csv(f'data/clean/data_motor_{idx}.csv')
    df_motor.insert(0, 'unit_number', idx)
    dfs.append(df_motor)
df_all = pd.concat(dfs, ignore_index=True)

pipeline = RULPipeline(window_size=30, clipping_threshold=125, verbose=True)
t0 = time.perf_counter()
motor_windows = pipeline.transform(df_all)
print(f"Tiempo total 140 motores: {time.perf_counter()-t0:.1f}s")

Training engines: 140
Test engines: 60
[Nodo 2] 140 motors, 19764 windows — 0.59s
[Nodo 3] 192 features per window — 378.84s
[Total]  379.43s
Tiempo total 140 motores: 379.4s


In [ ]:
import time
import pandas as pd
from src.pipeline.windowing import build_windows
from src.pipeline.feature_extraction import (
    STATISTICAL_FEATURES, TREND_FEATURES, extract_window_features
)

df = pd.read_csv('data/clean/data_motor_1.csv')
df.insert(0, 'unit_number', 1)
windows = build_windows(df, window_size=30, clipping_threshold=125)

t0 = time.perf_counter()
extract_window_features(windows, feature_config=STATISTICAL_FEATURES)
print(f"Solo STATISTICAL: {time.perf_counter()-t0:.2f}s")

t0 = time.perf_counter()
extract_window_features(windows, feature_config=TREND_FEATURES)
print(f"Solo TREND: {time.perf_counter()-t0:.2f}s")

In [ ]:
from src.dataset_manager import DatasetManager, store_dataframe_csv

# 1. Reconstruir dataset base corregido
df = DatasetManager.get_base_dataset()

# Verificar corrección
train_motors = df[df['evento'].isin([0,1])]
eventos_por_motor = df.groupby('unit_number')['evento'].sum()
print("Eventos por motor (debe ser 0 o 1):")
print(eventos_por_motor.value_counts())

# 2. Limpiar columnas constantes
df_clean, stats = DatasetManager.clean_dataset(df)

# 3. Guardar metadata
metadata = DatasetManager.generate_metadata(df_clean)
store_dataframe_csv(metadata, 'metadata', 'data')
print(f"\nMetadata guardada: {len(metadata)} motores")

# 4. Guardar CSV por motor
for unit_id in df_clean['unit_number'].unique():
    motor_df = df_clean[df_clean['unit_number'] == unit_id].drop(
        columns=['unit_number']
    ).reset_index(drop=True)
    store_dataframe_csv(motor_df, f'data_motor_{unit_id}', 'data/clean')

print(f"CSVs generados: {df_clean['unit_number'].nunique()} motores")

In [ ]:
# Guardar dataset unificado completo (sin filtrar)
store_dataframe_csv(df, 'dataset_unificado', 'data')
print("Dataset unificado guardado")

# Guardar también el limpio unificado por si acaso
store_dataframe_csv(df_clean, 'dataset_limpio', 'data')
print("Dataset limpio guardado")

In [ ]:
import pandas as pd

# Verificar un motor de train — debe tener evento=1 solo en el último ciclo
motor_train = pd.read_csv('data/clean/data_motor_1.csv')
print("Motor 1 (train):")
print(f"  Filas: {len(motor_train)}")
print(f"  Suma evento: {motor_train['evento'].sum()} (debe ser 1)")
print(f"  Evento en último ciclo: {motor_train['evento'].iloc[-1]} (debe ser 1)")
print(f"  Evento en primer ciclo: {motor_train['evento'].iloc[0]} (debe ser 0)")
print(f"  RUL último ciclo: {motor_train['RUL'].iloc[-1]} (debe ser 0)")

# Verificar un motor de test — debe tener evento=0 en todos los ciclos
motor_test = pd.read_csv('data/clean/data_motor_101.csv')
print("\nMotor 101 (test/censurado):")
print(f"  Filas: {len(motor_test)}")
print(f"  Suma evento: {motor_test['evento'].sum()} (debe ser 0)")
print(f"  RUL último ciclo: {motor_test['RUL'].iloc[-1]} (debe ser > 0)")

In [ ]:
import pandas as pd

metadata = pd.read_csv('data/metadata.csv')

# Debe haber exactamente 100 motores con event=1 y 100 con event=0
print(metadata['event'].value_counts())

# Verificar motor 1 específicamente
print(metadata[metadata['unit_number'] == 1])

In [ ]:
from src.dataset_manager import DatasetManager
from src.models.cox_frailty import CoxFrailty
from src.training_manager import GGSTrainingManager

m_train, m_test = DatasetManager.split_dataset()

Training = GGSTrainingManager(
    model=CoxFrailty(),
    list_ids=m_train
)

param_grid = {
    'distribution':         ['gamma', 'gaussian'],
    'confidence_threshold': [0.5, 0.7, 0.9],
    'clipping_threshold':   [110, 115, 120, 125]
}

ggs = Training.group_grid_search(
    param_grid=param_grid,
    n_folds=5,
    silence=True
)

display(Training.get_ggs_results(top_n=10))

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from src.models.cox_frailty import CoxFrailty
from src.mad_scaler import MADScaler
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline

X, y_surv, y_metrics, groups = Training.get_training_data()

gkf = GroupKFold(n_splits=5)
train_idx, val_idx = next(iter(gkf.split(X, y_surv, groups)))

model = CoxFrailty(distribution='gamma')
pipeline = Pipeline([('scaler', MADScaler()), ('model', model)])
pipeline.set_output(transform='pandas')

warnings.filterwarnings('error', category=RuntimeWarning)
try:
    pipeline.fit(X.iloc[train_idx], y_surv[train_idx], model__groups=groups[train_idx])
    print("is_fitted:", model.is_fitted_)
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {e}")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from src.models.cox_frailty import CoxFrailty
from src.mad_scaler import MADScaler
from sklearn.model_selection import GroupKFold

X, y_surv, y_metrics, groups = Training.get_training_data()

gkf = GroupKFold(n_splits=5)
train_idx, _ = next(iter(gkf.split(X, y_surv, groups)))

X_fold = X.iloc[train_idx]
y_fold = y_surv[train_idx]
g_fold = groups[train_idx]

scaler = MADScaler()
X_scaled = scaler.fit_transform(X_fold)

# Ver exactamente qué llega a coxph
evento = y_fold['evento'].astype(int)
t_start = y_fold['t_start'].astype(float)
t_stop = y_fold['t_stop'].astype(float)

print(f"Filas: {len(X_fold)}")
print(f"Motores únicos: {len(set(g_fold.tolist()))}")
print(f"Eventos totales: {evento.sum()}")
print(f"t_start range: {t_start.min()} - {t_start.max()}")
print(f"t_stop range: {t_stop.min()} - {t_stop.max()}")
print(f"Eventos en último ciclo de cada motor:")
import pandas as pd
df_check = pd.DataFrame({'motor': g_fold, 'evento': evento, 't_stop': t_stop})
print(df_check.groupby('motor')['evento'].sum().value_counts())

# Test

In [3]:
"""Pipeline validation script — Phase 1: single motor (motor 1).

Runs each pipeline node sequentially on motor 1 and prints shape,
content and timing information at each stage.

Run from the project root:
    python pipeline_validation.py
Or paste cells into a Jupyter notebook.
"""

import time
import numpy as np
import pandas as pd

from src.pipeline.scaling import FeatureScaler
from src.pipeline.windowing import build_windows, flatten_windows
from src.pipeline.feature_extraction import extract_window_features
from src.pipeline.dim_reduction import DimReducer

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
WINDOW_SIZE = 30
CLIPPING_THRESHOLD = 125
N_COMPONENTS = 10
MOTOR_ID = 1

print("=" * 60)
print("PIPELINE VALIDATION — Phase 1: Motor 1")
print("=" * 60)

# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------
df = pd.read_csv(f'data/clean/data_motor_{MOTOR_ID}.csv')
df.insert(0, 'unit_number', MOTOR_ID)

print(f"\nInput DataFrame:")
print(f"  Shape:   {df.shape}")
print(f"  Columns: {list(df.columns)}")
print(f"  Cycles:  {df['time_in_cycles'].min()} → {df['time_in_cycles'].max()}")
print(f"  Eventos: {df['evento'].sum()}")

# Separate X and targets before pipeline
X_df = df.drop(columns=['RUL', 'evento'])
y_df = df[['unit_number', 'time_in_cycles', 'RUL', 'evento']]

print(f"\nAfter GGS separation:")
print(f"  X_df shape:  {X_df.shape}  (enters pipeline)")
print(f"  y_df shape:  {y_df.shape}  (stays in GGS)")

PIPELINE VALIDATION — Phase 1: Motor 1

Input DataFrame:
  Shape:   (192, 20)
  Columns: ['unit_number', 'time_in_cycles', 'op_setting_1', 'op_setting_2', 'T24', 'T30', 'T50', 'P30', 'Nf', 'Nc', 'Ps30', 'phi', 'NRf', 'NRc', 'BPR', 'htBleed', 'W31', 'W32', 'RUL', 'evento']
  Cycles:  1 → 192
  Eventos: 1

After GGS separation:
  X_df shape:  (192, 18)  (enters pipeline)
  y_df shape:  (192, 4)  (stays in GGS)


In [4]:
# ---------------------------------------------------------------------------
# Nodo 1 — FeatureScaler
# ---------------------------------------------------------------------------
print("\n" + "-" * 60)
print("NODO 1 — FeatureScaler (RobustScaler)")
print("-" * 60)

t0 = time.perf_counter()
scaler = FeatureScaler()
scaler.fit(X_df)
X_scaled: pd.DataFrame = scaler.transform(X_df)
t1 = time.perf_counter()

print(f"  Input shape:    {X_df.shape}")
print(f"  Output shape:   {X_scaled.shape}")
print(f"  Scaled cols:    {scaler.feature_cols_[:3]} ... ({len(scaler.feature_cols_)} total)")
print(f"  unit_number unchanged: {(X_scaled['unit_number'] == X_df['unit_number']).all()}")
print(f"  time_in_cycles unchanged: {(X_scaled['time_in_cycles'] == X_df['time_in_cycles']).all()}")
print(f"  Sensor median after scaling (T24): {X_scaled['T24'].median():.6f} (should be ~0)")
print(f"  Time: {t1 - t0:.3f}s")


------------------------------------------------------------
NODO 1 — FeatureScaler (RobustScaler)
------------------------------------------------------------
  Input shape:    (192, 18)
  Output shape:   (192, 18)
  Scaled cols:    ['op_setting_1', 'op_setting_2', 'T24'] ... (16 total)
  unit_number unchanged: True
  time_in_cycles unchanged: True
  Sensor median after scaling (T24): 0.000000 (should be ~0)
  Time: 0.466s


In [5]:

# ---------------------------------------------------------------------------
# Nodo 2 — build_windows
# ---------------------------------------------------------------------------
print("\n" + "-" * 60)
print(f"NODO 2 — build_windows (window_size={WINDOW_SIZE})")
print("-" * 60)

t0 = time.perf_counter()
motor_windows = build_windows(X_scaled, window_size=WINDOW_SIZE, clipping_threshold=CLIPPING_THRESHOLD)
t1 = time.perf_counter()

data = motor_windows[MOTOR_ID]
print(f"  Motors in output:     {len(motor_windows)}")
print(f"  X_windows shape:      {data['X_windows'].shape}  (n_windows, window_size, n_sensors)")
print(f"  t_start range:        {data['t_start'].min():.0f} → {data['t_start'].max():.0f}")
print(f"  t_stop range:         {data['t_stop'].min():.0f} → {data['t_stop'].max():.0f}")
print(f"  t_start < t_stop:     {(data['t_start'] < data['t_stop']).all()}")
print(f"  Eventos en ventanas:  {data['evento'].sum()} (debe ser 0 — RUL/evento no entran)")
print(f"  y_rul NaN:            {np.isnan(data['y_rul']).all()} (esperado sin RUL en X_df)")
print(f"  feature_names[:3]:    {data['feature_names'][:3]}")
print(f"  Time: {t1 - t0:.3f}s")


------------------------------------------------------------
NODO 2 — build_windows (window_size=30)
------------------------------------------------------------
  Motors in output:     1
  X_windows shape:      (163, 30, 16)  (n_windows, window_size, n_sensors)
  t_start range:        0 → 162
  t_stop range:         30 → 192
  t_start < t_stop:     True
  Eventos en ventanas:  0 (debe ser 0 — RUL/evento no entran)
  y_rul NaN:            True (esperado sin RUL en X_df)
  feature_names[:3]:    ['op_setting_1', 'op_setting_2', 'T24']
  Time: 0.538s


In [6]:
# ---------------------------------------------------------------------------
# Nodo 3 — extract_window_features
# ---------------------------------------------------------------------------
print("\n" + "-" * 60)
print("NODO 3 — extract_window_features (numpy)")
print("-" * 60)

t0 = time.perf_counter()
motor_features = extract_window_features(motor_windows)
t1 = time.perf_counter()

data3 = motor_features[MOTOR_ID]
print(f"  Input X_windows shape:   {motor_windows[MOTOR_ID]['X_windows'].shape}")
print(f"  Output X_windows shape:  {data3['X_windows'].shape}  (n_windows, n_features)")
print(f"  n_features:              {data3['X_windows'].shape[1]}")
print(f"  feature_names[:3]:       {data3['feature_names'][:3]}")
print(f"  NaN in output:           {np.isnan(data3['X_windows']).any()}")
print(f"  Inf in output:           {not np.isfinite(data3['X_windows']).all()}")
print(f"  t_start unchanged:       {np.array_equal(data3['t_start'], motor_windows[MOTOR_ID]['t_start'])}")
print(f"  t_stop unchanged:        {np.array_equal(data3['t_stop'], motor_windows[MOTOR_ID]['t_stop'])}")
print(f"  Time: {t1 - t0:.3f}s")


------------------------------------------------------------
NODO 3 — extract_window_features (numpy)
------------------------------------------------------------
  Input X_windows shape:   (163, 30, 16)
  Output X_windows shape:  (163, 192)  (n_windows, n_features)
  n_features:              192
  feature_names[:3]:       ['op_setting_1__median', 'op_setting_2__median', 'T24__median']
  NaN in output:           False
  Inf in output:           False
  t_start unchanged:       True
  t_stop unchanged:        True
  Time: 4.601s


In [7]:
# ---------------------------------------------------------------------------
# Nodo 4 — DimReducer (PCA)
# ---------------------------------------------------------------------------
print("\n" + "-" * 60)
print(f"NODO 4 — DimReducer (PCA, n_components={N_COMPONENTS})")
print("-" * 60)

t0 = time.perf_counter()
reducer = DimReducer(n_components=N_COMPONENTS)
reducer.fit(motor_features)
motor_reduced = reducer.transform(motor_features)
t1 = time.perf_counter()

data4 = motor_reduced[MOTOR_ID]
evr = reducer.explained_variance_ratio()
print(f"  Input X_windows shape:   {motor_features[MOTOR_ID]['X_windows'].shape}")
print(f"  Output X_windows shape:  {data4['X_windows'].shape}  (n_windows, n_components)")
print(f"  feature_names:           {data4['feature_names']}")
print(f"  Explained variance:      {evr.round(3)}")
print(f"  Cumulative variance:     {evr.cumsum()[-1]:.3f}")
print(f"  NaN in output:           {np.isnan(data4['X_windows']).any()}")
print(f"  t_start unchanged:       {np.array_equal(data4['t_start'], motor_features[MOTOR_ID]['t_start'])}")
print(f"  t_stop unchanged:        {np.array_equal(data4['t_stop'], motor_features[MOTOR_ID]['t_stop'])}")
print(f"  Time: {t1 - t0:.3f}s")


------------------------------------------------------------
NODO 4 — DimReducer (PCA, n_components=10)
------------------------------------------------------------
  Input X_windows shape:   (163, 192)
  Output X_windows shape:  (163, 10)  (n_windows, n_components)
  feature_names:           ['PC_1', 'PC_2', 'PC_3', 'PC_4', 'PC_5', 'PC_6', 'PC_7', 'PC_8', 'PC_9', 'PC_10']
  Explained variance:      [0.951 0.02  0.012 0.007 0.003 0.002 0.001 0.001 0.001 0.   ]
  Cumulative variance:     0.998
  NaN in output:           False
  t_start unchanged:       True
  t_stop unchanged:        True
  Time: 0.323s


In [8]:
# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("SUMMARY — Motor 1")
print("=" * 60)
print(f"  Input:   DataFrame {df.shape}")
print(f"  Nodo 1:  {X_df.shape} → {X_scaled.shape}  (scaled)")
print(f"  Nodo 2:  {X_scaled.shape} → {motor_windows[MOTOR_ID]['X_windows'].shape}  (windows)")
print(f"  Nodo 3:  {motor_windows[MOTOR_ID]['X_windows'].shape} → {motor_features[MOTOR_ID]['X_windows'].shape}  (features)")
print(f"  Nodo 4:  {motor_features[MOTOR_ID]['X_windows'].shape} → {motor_reduced[MOTOR_ID]['X_windows'].shape}  (PCA)")


SUMMARY — Motor 1
  Input:   DataFrame (192, 20)
  Nodo 1:  (192, 18) → (192, 18)  (scaled)
  Nodo 2:  (192, 18) → (163, 30, 16)  (windows)
  Nodo 3:  (163, 30, 16) → (163, 192)  (features)
  Nodo 4:  (163, 192) → (163, 10)  (PCA)


In [ ]:
"""Pipeline validation script — Phase 1: single motor (motor 1).

Runs each pipeline node sequentially on motor 1 and prints shape,
content and timing information at each stage.

Run from the project root:
    python pipeline_validation.py
Or paste cells into a Jupyter notebook.
"""

import time
import numpy as np
import pandas as pd

from src.pipeline.scaling import FeatureScaler
from src.pipeline.windowing import build_windows, flatten_windows
from src.pipeline.feature_extraction import extract_window_features
from src.pipeline.dim_reduction import DimReducer

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
WINDOW_SIZE = 30
CLIPPING_THRESHOLD = 125
N_COMPONENTS = 10
MOTOR_ID = 1

print("=" * 60)
print("PIPELINE VALIDATION — Phase 1: Motor 1")
print("=" * 60)

# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------
df = pd.read_csv(f'data/clean/data_motor_{MOTOR_ID}.csv')
df.insert(0, 'unit_number', MOTOR_ID)

print(f"\nInput DataFrame:")
print(f"  Shape:   {df.shape}")
print(f"  Columns: {list(df.columns)}")
print(f"  Cycles:  {df['time_in_cycles'].min()} → {df['time_in_cycles'].max()}")
print(f"  Eventos: {df['evento'].sum()}")

X_df = df.drop(columns=['RUL'])  # ← solo RUL se separa
y_df = df[['unit_number', 'time_in_cycles', 'RUL']]  # ← solo RUL en y_df

print(f"\nAfter GGS separation:")
print(f"  X_df shape:  {X_df.shape}  (enters pipeline)")
print(f"  y_df shape:  {y_df.shape}  (stays in GGS)")

# ---------------------------------------------------------------------------
# Nodo 1 — FeatureScaler
# ---------------------------------------------------------------------------
print("\n" + "-" * 60)
print("NODO 1 — FeatureScaler (RobustScaler)")
print("-" * 60)

t0 = time.perf_counter()
scaler = FeatureScaler()
scaler.fit(X_df)
X_scaled: pd.DataFrame = scaler.transform(X_df)
t1 = time.perf_counter()

print(f"  Input shape:    {X_df.shape}")
print(f"  Output shape:   {X_scaled.shape}")
print(f"  Scaled cols:    {scaler.feature_cols_[:3]} ... ({len(scaler.feature_cols_)} total)")
print(f"  unit_number unchanged: {(X_scaled['unit_number'] == X_df['unit_number']).all()}")
print(f"  time_in_cycles unchanged: {(X_scaled['time_in_cycles'] == X_df['time_in_cycles']).all()}")
print(f"  Sensor median after scaling (T24): {X_scaled['T24'].median():.6f} (should be ~0)")
print(f"  Time: {t1 - t0:.3f}s")

# ---------------------------------------------------------------------------
# Nodo 2 — build_windows
# ---------------------------------------------------------------------------
print("\n" + "-" * 60)
print(f"NODO 2 — build_windows (window_size={WINDOW_SIZE})")
print("-" * 60)

t0 = time.perf_counter()
motor_windows = build_windows(X_scaled, window_size=WINDOW_SIZE, clipping_threshold=CLIPPING_THRESHOLD)
t1 = time.perf_counter()

data = motor_windows[MOTOR_ID]
print(f"  Motors in output:     {len(motor_windows)}")
print(f"  X_windows shape:      {data['X_windows'].shape}  (n_windows, window_size, n_sensors)")
print(f"  t_start range:        {data['t_start'].min():.0f} → {data['t_start'].max():.0f}")
print(f"  t_stop range:         {data['t_stop'].min():.0f} → {data['t_stop'].max():.0f}")
print(f"  t_start < t_stop:     {(data['t_start'] < data['t_stop']).all()}")
print(f"  Eventos en ventanas:  {data['evento'].sum()}")
print(f"  y_rul NaN:            {np.isnan(data['y_rul']).all()} (esperado sin RUL en X_df)")
print(f"  feature_names[:3]:    {data['feature_names'][:3]}")
print(f"  Time: {t1 - t0:.3f}s")

# ---------------------------------------------------------------------------
# Nodo 3 — extract_window_features
# ---------------------------------------------------------------------------
print("\n" + "-" * 60)
print("NODO 3 — extract_window_features (numpy)")
print("-" * 60)

t0 = time.perf_counter()
motor_features = extract_window_features(motor_windows)
t1 = time.perf_counter()

data3 = motor_features[MOTOR_ID]
print(f"  Input X_windows shape:   {motor_windows[MOTOR_ID]['X_windows'].shape}")
print(f"  Output X_windows shape:  {data3['X_windows'].shape}  (n_windows, n_features)")
print(f"  n_features:              {data3['X_windows'].shape[1]}")
print(f"  feature_names[:3]:       {data3['feature_names'][:3]}")
print(f"  NaN in output:           {np.isnan(data3['X_windows']).any()}")
print(f"  Inf in output:           {not np.isfinite(data3['X_windows']).all()}")
print(f"  t_start unchanged:       {np.array_equal(data3['t_start'], motor_windows[MOTOR_ID]['t_start'])}")
print(f"  t_stop unchanged:        {np.array_equal(data3['t_stop'], motor_windows[MOTOR_ID]['t_stop'])}")
print(f"  Time: {t1 - t0:.3f}s")

# ---------------------------------------------------------------------------
# Nodo 4 — DimReducer (PCA)
# ---------------------------------------------------------------------------
print("\n" + "-" * 60)
print(f"NODO 4 — DimReducer (PCA, n_components={N_COMPONENTS})")
print("-" * 60)

t0 = time.perf_counter()
reducer = DimReducer(n_components=N_COMPONENTS)
reducer.fit(motor_features)
motor_reduced = reducer.transform(motor_features)
t1 = time.perf_counter()

data4 = motor_reduced[MOTOR_ID]
evr = reducer.explained_variance_ratio()
print(f"  Input X_windows shape:   {motor_features[MOTOR_ID]['X_windows'].shape}")
print(f"  Output X_windows shape:  {data4['X_windows'].shape}  (n_windows, n_components)")
print(f"  feature_names:           {data4['feature_names']}")
print(f"  Explained variance:      {evr.round(3)}")
print(f"  Cumulative variance:     {evr.cumsum()[-1]:.3f}")
print(f"  NaN in output:           {np.isnan(data4['X_windows']).any()}")
print(f"  t_start unchanged:       {np.array_equal(data4['t_start'], motor_features[MOTOR_ID]['t_start'])}")
print(f"  t_stop unchanged:        {np.array_equal(data4['t_stop'], motor_features[MOTOR_ID]['t_stop'])}")
print(f"  Time: {t1 - t0:.3f}s")

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("SUMMARY — Motor 1")
print("=" * 60)
print(f"  Input:   DataFrame {df.shape}")
print(f"  Nodo 1:  {X_df.shape} → {X_scaled.shape}  (scaled)")
print(f"  Nodo 2:  {X_scaled.shape} → {motor_windows[MOTOR_ID]['X_windows'].shape}  (windows)")
print(f"  Nodo 3:  {motor_windows[MOTOR_ID]['X_windows'].shape} → {motor_features[MOTOR_ID]['X_windows'].shape}  (features)")
print(f"  Nodo 4:  {motor_features[MOTOR_ID]['X_windows'].shape} → {motor_reduced[MOTOR_ID]['X_windows'].shape}  (PCA)")

PIPELINE VALIDATION — Phase 1: Motor 1

Input DataFrame:
  Shape:   (192, 20)
  Columns: ['unit_number', 'time_in_cycles', 'op_setting_1', 'op_setting_2', 'T24', 'T30', 'T50', 'P30', 'Nf', 'Nc', 'Ps30', 'phi', 'NRf', 'NRc', 'BPR', 'htBleed', 'W31', 'W32', 'RUL', 'evento']
  Cycles:  1 → 192
  Eventos: 1

After GGS separation:
  X_df shape:  (192, 19)  (enters pipeline)
  y_df shape:  (192, 3)  (stays in GGS)

------------------------------------------------------------
NODO 1 — FeatureScaler (RobustScaler)
------------------------------------------------------------
  Input shape:    (192, 19)
  Output shape:   (192, 19)
  Scaled cols:    ['op_setting_1', 'op_setting_2', 'T24'] ... (16 total)
  unit_number unchanged: True
  time_in_cycles unchanged: True
  Sensor median after scaling (T24): 0.000000 (should be ~0)
  Time: 0.027s

------------------------------------------------------------
NODO 2 — build_windows (window_size=30)
---------------------------------------------------------

In [2]:
print(list(X_df.columns))

['unit_number', 'time_in_cycles', 'op_setting_1', 'op_setting_2', 'T24', 'T30', 'T50', 'P30', 'Nf', 'Nc', 'Ps30', 'phi', 'NRf', 'NRc', 'BPR', 'htBleed', 'W31', 'W32']


In [2]:
"""Pipeline validation script — Phase 2: full GGS fold simulation.

Simulates one iteration of the GGS loop using a real GroupKFold split
over the training partition. Validates shapes, target integrity, and
timing across all pipeline nodes for a complete fold.

Run from the project root:
    python pipeline_validation_phase2.py
Or paste cells into a Jupyter notebook.
"""

import time
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

from src.dataset_manager import DatasetManager
from src.pipeline.scaling import FeatureScaler
from src.pipeline.windowing import build_windows, flatten_windows
from src.pipeline.feature_extraction import extract_window_features
from src.pipeline.dim_reduction import DimReducer

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
WINDOW_SIZE = 30
CLIPPING_THRESHOLD = 125
N_COMPONENTS = 10
N_FOLDS = 5

print("=" * 60)
print("PIPELINE VALIDATION — Phase 2: Full GGS fold")
print("=" * 60)

# ---------------------------------------------------------------------------
# Load all training motors
# ---------------------------------------------------------------------------
t_load = time.perf_counter()
m_train, m_test = DatasetManager.split_dataset()

dfs = []
for idx in m_train:
    df_motor = pd.read_csv(f'data/clean/data_motor_{idx}.csv')
    df_motor.insert(0, 'unit_number', idx)
    dfs.append(df_motor)

df_all = pd.concat(dfs, ignore_index=True)
print(f"\nFull training DataFrame:")
print(f"  Motors:  {df_all['unit_number'].nunique()}")
print(f"  Rows:    {len(df_all)}")
print(f"  Eventos: {df_all['evento'].sum()}")
print(f"  Load time: {time.perf_counter() - t_load:.2f}s")

# GGS separation — done once before folds
X_df_full = df_all.drop(columns=['RUL'])
y_df_full = df_all[['unit_number', 'time_in_cycles', 'RUL']]
groups_full = df_all['unit_number'].to_numpy()

print(f"\nAfter GGS separation:")
print(f"  X_df shape: {X_df_full.shape}  (enters pipeline)")
print(f"  y_df shape: {y_df_full.shape}  (stays in GGS)")

# ---------------------------------------------------------------------------
# Simulate one fold
# ---------------------------------------------------------------------------
print(f"\n{'=' * 60}")
print(f"FOLD SIMULATION (fold 0 of {N_FOLDS})")
print("=" * 60)

gkf = GroupKFold(n_splits=N_FOLDS)
train_idx, val_idx = next(iter(gkf.split(X_df_full, groups=groups_full)))

X_train_df = X_df_full.iloc[train_idx].reset_index(drop=True)
X_val_df = X_df_full.iloc[val_idx].reset_index(drop=True)
y_train = y_df_full.iloc[train_idx].reset_index(drop=True)
y_val = y_df_full.iloc[val_idx].reset_index(drop=True)

print(f"\nFold split:")
print(f"  Train: {X_train_df['unit_number'].nunique()} motors, {len(X_train_df)} rows")
print(f"  Val:   {X_val_df['unit_number'].nunique()} motors, {len(X_val_df)} rows")

# ---------------------------------------------------------------------------
# Nodo 1 — FeatureScaler (fit on train, transform both)
# ---------------------------------------------------------------------------
print(f"\n{'-' * 60}")
print("NODO 1 — FeatureScaler")
print("-" * 60)

t0 = time.perf_counter()
scaler = FeatureScaler()
scaler.fit(X_train_df)
X_train_scaled: pd.DataFrame = scaler.transform(X_train_df)
X_val_scaled: pd.DataFrame = scaler.transform(X_val_df)
t1 = time.perf_counter()

print(f"  Train scaled shape: {X_train_scaled.shape}")
print(f"  Val scaled shape:   {X_val_scaled.shape}")
print(f"  T24 train median after scaling: {X_train_scaled['T24'].median():.6f}")
print(f"  Time: {t1 - t0:.3f}s")

# ---------------------------------------------------------------------------
# Nodo 2 — build_windows (train and val separately)
# ---------------------------------------------------------------------------
print(f"\n{'-' * 60}")
print(f"NODO 2 — build_windows (window_size={WINDOW_SIZE})")
print("-" * 60)

t0 = time.perf_counter()
train_windows = build_windows(X_train_scaled, window_size=WINDOW_SIZE,
                               clipping_threshold=CLIPPING_THRESHOLD)
val_windows = build_windows(X_val_scaled, window_size=WINDOW_SIZE,
                             clipping_threshold=CLIPPING_THRESHOLD)
t1 = time.perf_counter()

n_train_windows = sum(d['X_windows'].shape[0] for d in train_windows.values())
n_val_windows = sum(d['X_windows'].shape[0] for d in val_windows.values())
first_train = next(iter(train_windows.values()))

print(f"  Train: {len(train_windows)} motors, {n_train_windows} windows")
print(f"  Val:   {len(val_windows)} motors, {n_val_windows} windows")
print(f"  X_windows shape (one motor): {first_train['X_windows'].shape}")
print(f"  Time: {t1 - t0:.3f}s")

# ---------------------------------------------------------------------------
# Nodo 3 — extract_window_features
# ---------------------------------------------------------------------------
print(f"\n{'-' * 60}")
print("NODO 3 — extract_window_features")
print("-" * 60)

t0 = time.perf_counter()
train_features = extract_window_features(train_windows)
val_features = extract_window_features(val_windows)
t1 = time.perf_counter()

first_train_feat = next(iter(train_features.values()))
n_features = first_train_feat['X_windows'].shape[1]
n_train_feat_windows = sum(d['X_windows'].shape[0] for d in train_features.values())

print(f"  Train: {n_train_feat_windows} windows, {n_features} features")
print(f"  Val:   {sum(d['X_windows'].shape[0] for d in val_features.values())} windows")
print(f"  NaN in train: {any(np.isnan(d['X_windows']).any() for d in train_features.values())}")
print(f"  Time: {t1 - t0:.2f}s  ← main bottleneck")

# ---------------------------------------------------------------------------
# Nodo 4 — DimReducer (fit on train, transform both)
# ---------------------------------------------------------------------------
print(f"\n{'-' * 60}")
print(f"NODO 4 — DimReducer (n_components={N_COMPONENTS})")
print("-" * 60)

t0 = time.perf_counter()
reducer = DimReducer(n_components=N_COMPONENTS)
reducer.fit(train_features)
train_reduced = reducer.transform(train_features)
val_reduced = reducer.transform(val_features)
t1 = time.perf_counter()

evr = reducer.explained_variance_ratio()
first_train_red = next(iter(train_reduced.values()))

print(f"  Train reduced shape (one motor): {first_train_red['X_windows'].shape}")
print(f"  Explained variance: {evr.round(3)}")
print(f"  Cumulative variance: {evr.cumsum()[-1]:.3f}")
print(f"  Time: {t1 - t0:.3f}s")

# ---------------------------------------------------------------------------
# Flatten — ready for model
# ---------------------------------------------------------------------------
print(f"\n{'-' * 60}")
print("FLATTEN — ready for survival model")
print("-" * 60)

t0 = time.perf_counter()
X_train, t_start_tr, t_stop_tr, evento_tr, y_rul_tr, groups_tr = flatten_windows(train_reduced)
X_val, t_start_val, t_stop_val, evento_val, y_rul_val, groups_val = flatten_windows(val_reduced)
t1 = time.perf_counter()

print(f"  X_train shape:  {X_train.shape}  (n_windows, n_components)")
print(f"  X_val shape:    {X_val.shape}")
print(f"  groups_train unique motors: {len(np.unique(groups_tr))}")
print(f"  evento_train sum: {evento_tr.sum()} (all 0 — no RUL/evento in pipeline)")
print(f"  y_rul_train NaN: {np.isnan(y_rul_tr).all()} (expected)")
print(f"  Time: {t1 - t0:.3f}s")

# ---------------------------------------------------------------------------
# Total timing
# ---------------------------------------------------------------------------
print(f"\n{'=' * 60}")
print("TOTAL TIMING SUMMARY")
print("=" * 60)
print(f"  Nodo 1 (scaling):           fast")
print(f"  Nodo 2 (windowing):         fast")
print(f"  Nodo 3 (feature extraction): bottleneck — see above")
print(f"  Nodo 4 (PCA):               fast")
print(f"\nNote: y_rul and evento are NaN/0 because RUL and evento")
print(f"were separated before the pipeline. The GGS must align")
print(f"y_df targets with the windowed output using t_stop as key.")

PIPELINE VALIDATION — Phase 2: Full GGS fold
Training engines: 140
Test engines: 60

Full training DataFrame:
  Motors:  140
  Rows:    23824
  Eventos: 69
  Load time: 1.23s

After GGS separation:
  X_df shape: (23824, 19)  (enters pipeline)
  y_df shape: (23824, 3)  (stays in GGS)

FOLD SIMULATION (fold 0 of 5)

Fold split:
  Train: 112 motors, 19062 rows
  Val:   28 motors, 4762 rows

------------------------------------------------------------
NODO 1 — FeatureScaler
------------------------------------------------------------
  Train scaled shape: (19062, 19)
  Val scaled shape:   (4762, 19)
  T24 train median after scaling: 0.000000
  Time: 0.120s

------------------------------------------------------------
NODO 2 — build_windows (window_size=30)
------------------------------------------------------------
  Train: 112 motors, 15814 windows
  Val:   28 motors, 3950 windows
  X_windows shape (one motor): (123, 30, 16)
  Time: 1.757s

-----------------------------------------------